# Export Confidence Boost Checkpoints to ExecuTorch (.pte)

This notebook exports the 50, 55, and 60 epoch confidence boost checkpoints to ExecuTorch format for mobile deployment.

## Input
- Checkpoints from: `/content/drive/MyDrive/HazProML/models/confidence_boost_97class/`

## Output
- `yolox_confidence_boost_epoch50.pte`
- `yolox_confidence_boost_epoch55.pte`
- `yolox_confidence_boost_epoch60.pte`

## What's Special About These Models
These models were trained with confidence boost modifications:
- 2x objectness loss weight
- 20x lower learning rate
- 40 epochs no-augmentation phase

**Result:** Confidence ceiling broken from ~53% to 98%+

## Cell 1: Install Dependencies

In [1]:
!pip install executorch torch torchvision -q
!pip install opencv-python-headless numpy -q

import torch
import numpy as np
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.3/538.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB

## Cell 2: Mount Google Drive & Configuration

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# =============================================================================
# CONFIGURATION
# =============================================================================

# Source directory with confidence boost checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/HazProML/models/confidence_boost_97class'

# Checkpoints to export (epochs)
EPOCHS_TO_EXPORT = [50, 55, 60]

# Output directory for ExecuTorch models
OUTPUT_DIR = '/content/drive/MyDrive/HazProML/models/executorch_confidence_boost'

# Model configuration
NUM_CLASSES = 97
INPUT_SIZE = 640
DEPTH = 0.33  # YOLOX-Tiny
WIDTH = 0.375  # YOLOX-Tiny

# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Epochs to export: {EPOCHS_TO_EXPORT}")
print(f"Num classes: {NUM_CLASSES}")

# Check which checkpoints exist
print("\nAvailable checkpoints:")
for epoch in EPOCHS_TO_EXPORT:
    path = f"{CHECKPOINT_DIR}/confidence_boost_epoch_{epoch}.pth"
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1024 / 1024 if exists else 0
    status = f"{size:.1f} MB" if exists else "NOT FOUND"
    print(f"  Epoch {epoch}: {status}")

# Also check for final checkpoint
final_path = f"{CHECKPOINT_DIR}/confidence_boost_final.pth"
if os.path.exists(final_path):
    size = os.path.getsize(final_path) / 1024 / 1024
    print(f"  Final: {size:.1f} MB")

Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/HazProML/models/confidence_boost_97class
Output directory: /content/drive/MyDrive/HazProML/models/executorch_confidence_boost
Epochs to export: [50, 55, 60]
Num classes: 97

Available checkpoints:
  Epoch 50: 38.9 MB
  Epoch 55: 38.9 MB
  Epoch 60: 38.9 MB
  Final: 38.9 MB


## Cell 3: Setup YOLOX

In [3]:
import sys

if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

!pip install loguru thop ninja tabulate -q

%cd /content/YOLOX
sys.path.insert(0, '/content/YOLOX')

print("YOLOX setup complete!")

Cloning into '/content/YOLOX'...
remote: Enumerating objects: 1940, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 1940 (delta 5), reused 1 (delta 1), pack-reused 1924 (from 2)
Receiving objects: 100% (1940/1940), 7.55 MiB | 15.02 MiB/s, done.
Resolving deltas: 100% (1155/1155), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.3 MB/s eta 0:00:00
/content/YOLOX
YOLOX setup complete!


## Cell 4: Define Model Architecture

In [4]:
import torch
import torch.nn as nn
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

def create_yolox_tiny(num_classes):
    """Create YOLOX-Tiny model architecture."""
    in_channels = [256, 512, 1024]

    backbone = YOLOPAFPN(
        depth=DEPTH,
        width=WIDTH,
        in_channels=in_channels,
        act='silu'
    )

    head = YOLOXHead(
        num_classes=num_classes,
        width=WIDTH,
        in_channels=in_channels,
        act='silu'
    )

    model = YOLOX(backbone, head)
    return model

class YOLOXExportWrapper(nn.Module):
    """Wrapper for YOLOX model export to ExecuTorch."""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.model.eval()
        if hasattr(self.model.head, 'decode_in_inference'):
            self.model.head.decode_in_inference = False

    def forward(self, x):
        return self.model(x)

print(f"Model architecture defined: YOLOX-Tiny with {NUM_CLASSES} classes")

Model architecture defined: YOLOX-Tiny with 97 classes


## Cell 5: Export All Checkpoints

In [5]:
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

exported_files = []

for epoch in EPOCHS_TO_EXPORT:
    print("="*60)
    print(f"EXPORTING CONFIDENCE BOOST EPOCH {epoch}")
    print("="*60)

    checkpoint_path = f"{CHECKPOINT_DIR}/confidence_boost_epoch_{epoch}.pth"
    output_filename = f"yolox_confidence_boost_epoch{epoch}.pte"
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    if not os.path.exists(checkpoint_path):
        print(f"  SKIPPED: Checkpoint not found at {checkpoint_path}")
        continue

    try:
        # Create fresh model
        print(f"  Creating model...")
        model = create_yolox_tiny(num_classes=NUM_CLASSES)

        # Load checkpoint
        print(f"  Loading checkpoint...")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
        model.load_state_dict(state_dict, strict=False)
        model.eval()

        # Create export wrapper
        export_model = YOLOXExportWrapper(model)
        export_model.eval()

        # Test forward pass
        dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
        with torch.no_grad():
            test_output = export_model(dummy_input)
        print(f"  Output shape: {test_output.shape}")

        # Export
        print(f"  Exporting with torch.export...")
        example_input = (torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE),)
        exported_program = export(export_model, example_input)

        print(f"  Lowering to edge with XNNPACK...")
        try:
            edge_program = to_edge_transform_and_lower(
                exported_program,
                partitioner=[XnnpackPartitioner()]
            )
        except Exception as e:
            print(f"  XNNPACK failed, using basic edge: {e}")
            from executorch.exir import to_edge
            edge_program = to_edge(exported_program)

        print(f"  Converting to ExecuTorch...")
        executorch_program = edge_program.to_executorch()

        print(f"  Saving .pte file...")
        with open(output_path, 'wb') as f:
            f.write(executorch_program.buffer)

        file_size = os.path.getsize(output_path) / (1024 * 1024)
        print(f"  SUCCESS: {output_filename} ({file_size:.2f} MB)")
        exported_files.append((epoch, output_filename, file_size))

    except Exception as e:
        print(f"  FAILED: {e}")
        import traceback
        traceback.print_exc()

    print()

EXPORTING CONFIDENCE BOOST EPOCH 50
  Creating model...
  Loading checkpoint...
  Output shape: torch.Size([1, 8400, 102])
  Exporting with torch.export...
  Lowering to edge with XNNPACK...
  Converting to ExecuTorch...
  Saving .pte file...
  SUCCESS: yolox_confidence_boost_epoch50.pte (19.36 MB)

EXPORTING CONFIDENCE BOOST EPOCH 55
  Creating model...
  Loading checkpoint...
  Output shape: torch.Size([1, 8400, 102])
  Exporting with torch.export...
  Lowering to edge with XNNPACK...
  Converting to ExecuTorch...
  Saving .pte file...
  SUCCESS: yolox_confidence_boost_epoch55.pte (19.36 MB)

EXPORTING CONFIDENCE BOOST EPOCH 60
  Creating model...
  Loading checkpoint...
  Output shape: torch.Size([1, 8400, 102])
  Exporting with torch.export...
  Lowering to edge with XNNPACK...
  Converting to ExecuTorch...
  Saving .pte file...
  SUCCESS: yolox_confidence_boost_epoch60.pte (19.36 MB)



## Cell 6: Create Metadata Files

In [6]:
import json

for epoch, filename, size in exported_files:
    metadata = {
        "model_name": f"yolox_confidence_boost_epoch{epoch}",
        "model_type": "YOLOX-Tiny",
        "training_type": "confidence_boost_finetune",
        "base_training_epochs": 100,
        "finetune_epochs": epoch,
        "total_effective_epochs": 100 + epoch,
        "confidence_boost_settings": {
            "obj_loss_weight": 2.0,
            "learning_rate_multiplier": 0.05,
            "no_aug_epochs": 40
        },
        "num_classes": NUM_CLASSES,
        "input_size": INPUT_SIZE,
        "input_format": "NCHW",
        "input_channels": "BGR",
        "input_range": "0-255",
        "output_format": "[batch, anchors, 5+classes]",
        "output_shape": [1, 8400, 5 + NUM_CLASSES],
        "anchors": {
            "total": 8400,
            "strides": [8, 16, 32],
            "grid_sizes": [80, 40, 20]
        },
        "preprocessing": {
            "letterbox": True,
            "pad_value": 114,
            "normalize": False
        },
        "checkpoint_source": f"{CHECKPOINT_DIR}/confidence_boost_epoch_{epoch}.pth",
        "export_format": "ExecuTorch",
        "backend": "XNNPACK",
        "file_size_mb": round(size, 2),
        "expected_max_confidence": "98%+"
    }

    metadata_path = os.path.join(OUTPUT_DIR, f"yolox_confidence_boost_epoch{epoch}_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Created: {metadata_path}")

# Copy class mapping from combined_dataset
class_mapping_paths = [
    '/content/drive/MyDrive/HazProML/data/combined_dataset/class_mapping.json',
    f"{CHECKPOINT_DIR}/class_mapping.json"
]

for src in class_mapping_paths:
    if os.path.exists(src):
        import shutil
        shutil.copy(src, f"{OUTPUT_DIR}/class_mapping.json")
        print(f"Copied class_mapping.json from {src}")
        break

Created: /content/drive/MyDrive/HazProML/models/executorch_confidence_boost/yolox_confidence_boost_epoch50_metadata.json
Created: /content/drive/MyDrive/HazProML/models/executorch_confidence_boost/yolox_confidence_boost_epoch55_metadata.json
Created: /content/drive/MyDrive/HazProML/models/executorch_confidence_boost/yolox_confidence_boost_epoch60_metadata.json
Copied class_mapping.json from /content/drive/MyDrive/HazProML/data/combined_dataset/class_mapping.json


## Cell 7: Summary

In [7]:
print("="*60)
print("EXPORT COMPLETE!")
print("="*60)
print(f"\nExported {len(exported_files)} confidence boost models to:")
print(f"  {OUTPUT_DIR}")
print("\nFiles created:")
for epoch, filename, size in exported_files:
    print(f"  - {filename} ({size:.2f} MB)")

print("\n" + "="*60)
print("CONFIDENCE BOOST IMPROVEMENTS")
print("="*60)
print("""
These models have:
  - Confidence ceiling BROKEN (was ~53%, now 98%+)
  - 2x objectness loss weight
  - Fine-tuned for 60 epochs on top of 100-epoch base
  - Total effective training: 160 epochs
""")

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print(f"""
1. Download .pte files from Google Drive:
   {OUTPUT_DIR}/

2. Copy to React Native project:
   HazProML/assets/models/yolox_confidence_boost_epoch60.pte

3. Update executorchService.ts:
   const MODEL_PATH = 'yolox_confidence_boost_epoch60.pte';

4. Update src/types/detection.ts:
   numClasses: {NUM_CLASSES}

5. Make sure configs/class_mapping.json has 97 classes

6. Rebuild the app:
   npx expo run:ios
   npx expo run:android

RECOMMENDED: Start with epoch 60 (best confidence scores)
""")

# List all files in output directory
print("\nAll files in output directory:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f} ({size:.2f} MB)")

EXPORT COMPLETE!

Exported 3 confidence boost models to:
  /content/drive/MyDrive/HazProML/models/executorch_confidence_boost

Files created:
  - yolox_confidence_boost_epoch50.pte (19.36 MB)
  - yolox_confidence_boost_epoch55.pte (19.36 MB)
  - yolox_confidence_boost_epoch60.pte (19.36 MB)

CONFIDENCE BOOST IMPROVEMENTS

These models have:
  - Confidence ceiling BROKEN (was ~53%, now 98%+)
  - 2x objectness loss weight
  - Fine-tuned for 60 epochs on top of 100-epoch base
  - Total effective training: 160 epochs


NEXT STEPS

1. Download .pte files from Google Drive:
   /content/drive/MyDrive/HazProML/models/executorch_confidence_boost/

2. Copy to React Native project:
   HazProML/assets/models/yolox_confidence_boost_epoch60.pte

3. Update executorchService.ts:
   const MODEL_PATH = 'yolox_confidence_boost_epoch60.pte';

4. Update src/types/detection.ts:
   numClasses: 97

5. Make sure configs/class_mapping.json has 97 classes

6. Rebuild the app:
   npx expo run:ios
   npx expo run: